# Análise de dados eleitorais

Este notebook reúne todo o projeto da Aula 02 - Semana 05 em um único fluxo executável.

O projeto usa dados fictícios de eleições para praticar leitura de CSV, resumo, amostragem, agregação por partido, identificação de vencedores e exportação de resultados.

## Etapas do projeto

1. Carregar `dados/eleicoes.csv` preservando os zeros à esquerda de `zona` e `secao`.
2. Calcular um resumo geral dos dados.
3. Exibir uma amostra dos registros.
4. Somar e ordenar os votos por partido.
5. Identificar o candidato mais votado para prefeito em cada cidade.
6. Exportar os vencedores para `dados/resultado_prefeitos.csv`.
7. Apresentar um resumo consolidado de todos os resultados da análise.

In [1]:
from pathlib import Path
from IPython.display import display
import pandas as pd

CAMINHO_DADOS = Path("dados/eleicoes.csv")
CAMINHO_RESULTADO = Path("dados/resultado_prefeitos.csv")

## 1. Carregamento dos dados

As colunas `zona` e `secao` são lidas como texto para que valores como `005` não sejam convertidos para `5`.

In [2]:
df = pd.read_csv(CAMINHO_DADOS, dtype={"zona": str, "secao": str})
print(f"Arquivo: {CAMINHO_DADOS}")
print(f"Registros carregados: {len(df):,}".replace(",", "."))
display(df.head())
display(df.dtypes.rename("tipo").to_frame())

Arquivo: dados\eleicoes.csv
Registros carregados: 32


,ano,estado,cidade,zona,secao,cargo,candidato,partido,numero,votos
0,2024,SP,Campinas,101,0001,Prefeito,Ana Ribeiro,PFA,11,412
1,2024,SP,Campinas,101,0001,Prefeito,Bruno Tavares,PNB,22,388
2,2024,SP,Campinas,101,0001,Prefeito,Carla Mendes,PVC,33,157
3,2024,SP,Campinas,101,0002,Prefeito,Ana Ribeiro,PFA,11,365
4,2024,SP,Campinas,101,0002,Prefeito,Bruno Tavares,PNB,22,401


,tipo
ano,int64
estado,str
cidade,str
zona,str
secao,str
cargo,str
candidato,str
partido,str
numero,int64
votos,int64


## 2. Resumo geral

O resumo mostra a quantidade de registros, o total de votos e a quantidade de cidades e candidatos distintos.

In [3]:
resumo = pd.DataFrame(
    {
        "Indicador": ["Registros", "Votos", "Cidades", "Candidatos"],
        "Valor": [
            len(df),
            df["votos"].sum(),
            df["cidade"].nunique(),
            df["candidato"].nunique(),
        ],
    }
)
display(resumo)

,Indicador,Valor
0,Registros,32
1,Votos,9100
2,Cidades,5
3,Candidatos,16


## 3. Amostra dos dados

A tabela abaixo apresenta as cinco primeiras linhas e as colunas mais importantes para a análise.

In [4]:
colunas_amostra = [
    "estado", "cidade", "secao", "cargo", "candidato", "partido", "votos"
]
display(df[colunas_amostra].head(5))

,estado,cidade,secao,cargo,candidato,partido,votos
0,SP,Campinas,0001,Prefeito,Ana Ribeiro,PFA,412
1,SP,Campinas,0001,Prefeito,Bruno Tavares,PNB,388
2,SP,Campinas,0001,Prefeito,Carla Mendes,PVC,157
3,SP,Campinas,0002,Prefeito,Ana Ribeiro,PFA,365
4,SP,Campinas,0002,Prefeito,Bruno Tavares,PNB,401


## 4. Votos por partido

Os votos são agrupados por partido, ordenados do maior para o menor resultado e acompanhados do percentual sobre o total.

In [5]:
votos_por_partido = (
    df.groupby("partido")["votos"]
    .sum()
    .sort_values(ascending=False)
    .rename("votos")
    .to_frame()
)
votos_por_partido["percentual"] = (
    votos_por_partido["votos"] / votos_por_partido["votos"].sum() * 100
).round(1)
display(votos_por_partido)

,votos,percentual
partido,,
PFA,3603,39.6
PVC,2915,32.0
PNB,2582,28.4


## 5. Prefeitos vencedores por cidade

Primeiro são mantidas apenas as linhas do cargo `Prefeito`. Depois, os votos são somados por candidato em cada cidade. O vencedor é o candidato com maior total de votos.

In [6]:
prefeitos = df[df["cargo"] == "Prefeito"]

votos_por_candidato = (
    prefeitos.groupby(
        ["estado", "cidade", "candidato", "partido"], as_index=False
    )["votos"]
    .sum()
)
total_da_cidade = votos_por_candidato.groupby("cidade")["votos"].transform("sum")
votos_por_candidato["percentual"] = (
    votos_por_candidato["votos"] / total_da_cidade * 100
).round(1)

vencedores = (
    votos_por_candidato.sort_values("votos", ascending=False)
    .drop_duplicates("cidade")
    .sort_values("cidade")
    .reset_index(drop=True)
)

display(vencedores)

,estado,cidade,candidato,partido,votos,percentual
0,SP,Campinas,Bruno Tavares,PNB,789,41.6
1,PE,Olinda,Iara Santos,PNB,568,51.1
2,PE,Recife,Fabio Lins,PVC,1008,41.9
3,SP,Santos,Elisa Moura,PNB,668,52.1
4,MG,Uberlandia,Karina Alves,PFA,843,50.6


## 6. Exportação do resultado

O arquivo é salvo com `;` como separador, vírgula como separador decimal e codificação `utf-8-sig`, facilitando a abertura no Excel em português.

In [7]:
vencedores.to_csv(
    CAMINHO_RESULTADO,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

print(f"Resultado salvo em: {CAMINHO_RESULTADO}")
resultado_exportado = pd.read_csv(CAMINHO_RESULTADO, sep=";", decimal=",", encoding="utf-8-sig")
display(resultado_exportado)

Resultado salvo em: dados\resultado_prefeitos.csv


,estado,cidade,candidato,partido,votos,percentual
0,SP,Campinas,Bruno Tavares,PNB,789,41.6
1,PE,Olinda,Iara Santos,PNB,568,51.1
2,PE,Recife,Fabio Lins,PVC,1008,41.9
3,SP,Santos,Elisa Moura,PNB,668,52.1
4,MG,Uberlandia,Karina Alves,PFA,843,50.6


## 7. Resumo consolidado de todos os resultados

Abaixo reunimos a visão executiva completa de todas as etapas da análise: os indicadores gerais da eleição, o desempenho comparativo dos partidos (votos totais vs. prefeituras conquistadas) e a relação final de prefeitos eleitos.

In [8]:
# 1. Indicadores gerais da eleição
resumo_indicadores = pd.DataFrame(
    [
        {"Indicador": "Total de Seções / Registros", "Valor": f"{len(df):,}".replace(",", ".")},
        {"Indicador": "Total de Votos Apurados", "Valor": f"{df['votos'].sum():,}".replace(",", ".")},
        {"Indicador": "Municípios Analisados", "Valor": str(df["cidade"].nunique())},
        {"Indicador": "Candidatos Participantes", "Valor": str(df["candidato"].nunique())},
        {"Indicador": "Prefeituras em Disputa", "Valor": str(len(vencedores))},
    ]
)

# 2. Resumo consolidado por partido (Votos, Percentual e Prefeituras conquistadas)
prefeituras_partido = vencedores["partido"].value_counts().rename("prefeituras_eleitas")
cidades_partido = vencedores.groupby("partido")["cidade"].apply(lambda c: ", ".join(c)).rename("cidades_vencidas")

resumo_partidos = (
    votos_por_partido.copy()
    .join(prefeituras_partido)
    .fillna({"prefeituras_eleitas": 0})
    .join(cidades_partido)
    .fillna({"cidades_vencidas": "-"})
)
resumo_partidos["prefeituras_eleitas"] = resumo_partidos["prefeituras_eleitas"].astype(int)
resumo_partidos["%_prefeituras"] = (
    resumo_partidos["prefeituras_eleitas"] / len(vencedores) * 100
).round(1)

# 3. Prefeitos eleitos (visão final ordenada por cidade)
resumo_vencedores = vencedores[["estado", "cidade", "candidato", "partido", "votos", "percentual"]].copy()

print("--- 1. INDICADORES GERAIS ---")
display(resumo_indicadores)

print("\n--- 2. DESEMPENHO CONSOLIDADO POR PARTIDO ---")
display(resumo_partidos)

print("\n--- 3. RELAÇÃO FINAL DE PREFEITOS ELEITOS ---")
display(resumo_vencedores)

--- 1. INDICADORES GERAIS ---


,Indicador,Valor
0,Total de Seções / Registros,32
1,Total de Votos Apurados,9.100
2,Municípios Analisados,5
3,Candidatos Participantes,16
4,Prefeituras em Disputa,5



--- 2. DESEMPENHO CONSOLIDADO POR PARTIDO ---


,votos,percentual,prefeituras_eleitas,cidades_vencidas,%_prefeituras
partido,,,,,
PFA,3603,39.6,1,Uberlandia,20.0
PVC,2915,32.0,1,Recife,20.0
PNB,2582,28.4,3,"Campinas, Olinda, Santos",60.0



--- 3. RELAÇÃO FINAL DE PREFEITOS ELEITOS ---


,estado,cidade,candidato,partido,votos,percentual
0,SP,Campinas,Bruno Tavares,PNB,789,41.6
1,PE,Olinda,Iara Santos,PNB,568,51.1
2,PE,Recife,Fabio Lins,PVC,1008,41.9
3,SP,Santos,Elisa Moura,PNB,668,52.1
4,MG,Uberlandia,Karina Alves,PFA,843,50.6


### Síntese dos Resultados e Destaques

| Categoria | Destaque | Detalhe |
| :--- | :--- | :--- |
| **Volume Total de Votos** | **9.100 votos** | Apurados em 32 seções de 5 cidades nos estados de SP, PE e MG. |
| **Partido Mais Votado no Geral** | **PFA** (3.603 votos / 39,6%) | Obteve o maior volume absoluto de votos, porém elegeu apenas 1 prefeitura. |
| **Partido Mais Vitorioso em Prefeituras** | **PNB** (3 prefeituras / 60%) | Elegeu a maioria dos prefeitos (Campinas, Olinda e Santos), mesmo sendo o 3º em votos totais (28,4%). |
| **Maior Votação Nominal** | **Fabio Lins (PVC)** | 1.008 votos no município de Recife (PE). |
| **Maior Percentual em uma Cidade** | **Elisa Moura (PNB)** | 52,1% dos votos válidos em Santos (SP). |

#### Tabela Consolidada de Prefeitos Eleitos

| UF | Município | Prefeito(a) Eleito(a) | Partido | Votos Válidos | % na Cidade |
| :-: | :--- | :--- | :-: | :---: | :---: |
| **SP** | Campinas | Bruno Tavares | PNB | 789 | 41,6% |
| **PE** | Olinda | Iara Santos | PNB | 568 | 51,1% |
| **PE** | Recife | Fabio Lins | PVC | 1.008 | 41,9% |
| **SP** | Santos | Elisa Moura | PNB | 668 | 52,1% |
| **MG** | Uberlândia | Karina Alves | PFA | 843 | 50,6% |

## Conclusão

O fluxo completo foi executado com sucesso: os dados foram carregados preservando códigos de zona/seção, sintetizados em indicadores gerais, agregados por legenda partidária, filtrados para apuração dos vencedores majoritários e exportados para `dados/resultado_prefeitos.csv`.

O resumo consolidado evidencia a dinâmica das eleições: embora o **PFA** tenha obtido a maior parcela do total de votos (39,6%), o **PNB** conquistou a maior quantidade de prefeituras (60% dos municípios analisados), demonstrando a importância de analisar os resultados tanto pelo voto global quanto pela distribuição territorial.

> **Nota:** Todos os dados são fictícios e foram utilizados exclusivamente para fins didáticos no módulo de Análise de Dados com Python.